In [1]:
!pip install faker pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.2 MB/s eta 0:00:00


Adding randomness and creating template for datasets

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

random.seed(42)
np.random.seed(42)
Faker.seed(42)

CITIES = ["Bangalore", "Mumbai", "Delhi", "Chennai", "Hyderabad", "Pune", "Kolkata"]

def generate_clean_data(n_rows=50):
    data = []

    for _ in range(n_rows):
        row = {
            "name": fake.name(),
            "age": random.randint(18, 70),
            "email": fake.email(),
            "city": random.choice(CITIES),
            "salary": round(random.uniform(20000, 100000), 2)
        }
        data.append(row)

    df = pd.DataFrame(data)

    return df

In [3]:
df_clean = generate_clean_data(20)

df_clean.head()

,name,age,email,city,salary
0,Allison Hill,58,donaldgarcia@example.net,Bangalore,22000.86
1,Angie Henderson,35,davisjesse@example.net,Mumbai,37856.86
2,Cristian Santos,65,lrobinson@example.com,Bangalore,74135.96
3,Abigail Shaffer,52,jpeterson@example.org,Bangalore,67239.40
4,Gabrielle Davis,20,howardmaurice@example.com,Bangalore,27495.62


Checking dataset before corruption

In [4]:
print("Missing values:\n", df_clean.isnull().sum())
print("\nDuplicates:", df_clean.duplicated().sum())
print("\nData types:\n", df_clean.dtypes)

Missing values:
 name      0
age       0
email     0
city      0
salary    0
dtype: int64

Duplicates: 0

Data types:
 name       object
age         int64
email      object
city       object
salary    float64
dtype: object


Corrupting the dataset

In [5]:
def inject_missing_values(df, fraction=0.1):
    df = df.copy()

    n_rows, n_cols = df.shape
    total_cells = n_rows * n_cols
    n_missing = int(total_cells * fraction)

    for _ in range(n_missing):
        i = random.randint(0, n_rows - 1)
        j = random.randint(0, n_cols - 1)
        df.iat[i, j] = np.nan

    return df

In [6]:
def inject_duplicates(df, fraction=0.1):
    df = df.copy()

    n_rows = df.shape[0]
    n_duplicates = int(n_rows * fraction)

    duplicate_rows = df.sample(n=n_duplicates, replace=True)

    df = pd.concat([df, duplicate_rows], ignore_index=True)

    return df

In [13]:
def corrupt_salary_format(df, fraction=0.3):
    df = df.copy()

    # Convert entire column to object first (IMPORTANT)
    df["salary"] = df["salary"].astype(object)

    indices = df.sample(frac=fraction).index

    for i in indices:
        val = df.at[i, "salary"]
        df.at[i, "salary"] = f"${val:,.2f}"

    return df

In [14]:
def corrupt_dataset(df):
    df_dirty = df.copy()

    df_dirty = inject_missing_values(df_dirty, 0.1)
    df_dirty = inject_duplicates(df_dirty, 0.1)
    df_dirty = corrupt_salary_format(df_dirty, 0.3)

    return df_dirty

Checking the dataset after corruption

In [15]:
df_dirty = corrupt_dataset(df_clean)

df_dirty.head(10)

,name,age,email,city,salary
0,Allison Hill,58.0,donaldgarcia@example.net,Bangalore,22000.86
1,Angie Henderson,35.0,davisjesse@example.net,Mumbai,"$37,856.86"
2,Cristian Santos,65.0,lrobinson@example.com,Bangalore,74135.96
3,Abigail Shaffer,52.0,jpeterson@example.org,Bangalore,"$67,239.40"
4,Gabrielle Davis,20.0,howardmaurice@example.com,Bangalore,"$27,495.62"
5,Monica Herrera,32.0,smiller@example.net,Hyderabad,"$68,161.50"
6,Shannon Ray,53.0,NaN,Mumbai,77281.57
7,Dr. Sharon James,62.0,NaN,Hyderabad,NaN
8,Daniel Adams,46.0,lynchgeorge@example.net,Hyderabad,42255.26
9,Joel Nelson,18.0,gabriellecameron@example.org,NaN,84465.54


In [16]:
print("Missing values:\n", df_dirty.isnull().sum())
print("\nDuplicates:", df_dirty.duplicated().sum())
print("\nData types:\n", df_dirty.dtypes)

Missing values:
 name      1
age       2
email     3
city      1
salary    2
dtype: int64

Duplicates: 1

Data types:
 name       object
age       float64
email      object
city       object
salary     object
dtype: object


Function that will be called during reset

In [17]:
def generate_episode(n_rows=50):
    # Step 1: generate clean dataset
    df_clean = generate_clean_data(n_rows)

    # Step 2: corrupt it
    df_dirty = corrupt_dataset(df_clean)

    return df_clean, df_dirty

In [18]:
for i in range(3):
    clean, dirty = generate_episode(20)

    print(f"\n--- Episode {i+1} ---")
    print("Clean shape:", clean.shape)
    print("Dirty shape:", dirty.shape)
    print("Duplicates:", dirty.duplicated().sum())


--- Episode 1 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 0

--- Episode 2 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 1

--- Episode 3 ---
Clean shape: (20, 5)
Dirty shape: (22, 5)
Duplicates: 1


In [20]:
def run_pipeline(num_runs=5):
    for i in range(num_runs):
        clean, dirty = generate_episode(20)

        print(f"\n=== Run {i+1} ===")
        print("Dirty dataset preview:")
        print(dirty.head(3))
run_pipeline(3)


=== Run 1 ===
Dirty dataset preview:
              name  age                  email       city    salary
0  Patricia Becker   25                    NaN        NaN  30667.13
1  Richard Johnson   47  jessica14@example.com       Pune  62490.55
2   Casey Anderson   53   alyssa42@example.com  Hyderabad  45383.84

=== Run 2 ===
Dirty dataset preview:
                 name   age                    email       city    salary
0      Judith Maynard  26.0  lutzmelanie@example.com  Hyderabad  62729.89
1  Katherine Browning  43.0      lammarc@example.org  Hyderabad  65151.36
2      Donald Schultz  19.0      oscar98@example.com  Bangalore      $nan

=== Run 3 ===
Dirty dataset preview:
            name   age                    email     city    salary
0            NaN  59.0       tberry@example.net    Delhi  21012.17
1    Brian Smith  44.0     george97@example.net  Kolkata  59216.14
2  Anita Richard  45.0  ryanmorales@example.org    Delhi  70846.81


testing

In [21]:
generate_episode()

(                    name  age                         email       city  \
 0           James Martin   35  jenniferwilliams@example.com      Delhi   
 1        Cynthia Wallace   25           karen73@example.net       Pune   
 2        Vincent Mueller   25  michealvalentine@example.com       Pune   
 3          Douglas Reyes   66            mburch@example.net       Pune   
 4         David Davidson   31          angela18@example.org       Pune   
 5         Phillip Nelson   64       mariahdavis@example.org  Hyderabad   
 6           Stacey Arias   51            gbrown@example.com  Hyderabad   
 7         Zachary Brooks   24        florescory@example.net    Kolkata   
 8           Raven Taylor   32        samantha72@example.net      Delhi   
 9           Luis Bullock   18         scottmary@example.net       Pune   
 10        Lisa Henderson   35            john10@example.org  Bangalore   
 11          Riley Bryant   53   copelandvincent@example.net      Delhi   
 12            Jose Allen